# MedGemma Medication NER on MIMIC-IV — Colab Runner (T4 free tier)

Zero-shot evaluation of **google/medgemma-4b-it** on medication NER over real
**MIMIC-IV-Note** discharge summaries, scored with seqeval over six i2b2-style
types (Medication, Dose, Mode, Frequency, Duration, Reason).

**Before running:** `Runtime -> Change runtime type -> T4 GPU`.

---

## ⚠️ THIS NOTEBOOK HANDLES REAL PATIENT DATA

MIMIC-IV is credentialed PhysioNet data. Rules for this notebook:

- The sample file is **uploaded by hand** (cell 5). This notebook does **not**
  mount Google Drive and does **not** clone or download any note data.
- **Clear all outputs before saving or sharing this notebook.** Cell outputs can
  contain note text if you add your own inspection cells.
- Do not add cells that print note text, and do not commit the uploaded
  `.jsonl` anywhere.
- The metrics CSV and markdown report are aggregate-only and safe to keep.

Build the sample **on your own machine** first, where the credentialed data
lives:

```bash
python -m src.build_mimic_sample     # -> data/samples/mimic_med_sample.jsonl (~1.5 MB)
```

That file holds only the 100 sampled notes plus their gold spans. The 1.1 GB
`discharge.csv.gz` never leaves your machine.

## 1. Confirm the T4 GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU. Set Runtime -> Change runtime type -> T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))

## 2. Install dependencies

No `datasets`/`bioc` needed here — this evaluation reads a local JSONL rather
than a Hub dataset.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes seqeval pandas tqdm huggingface_hub

## 3. Hugging Face login (gated model)

Accept the license at https://huggingface.co/google/medgemma-4b-it, then paste a
token from https://huggingface.co/settings/tokens. Read via `getpass`, so it is
never stored in the notebook.

In [ ]:
from getpass import getpass
from huggingface_hub import login
login(getpass('HF token (input hidden): '))

## 4. Get the project code

Code only — the repo is public and contains **no** patient data.

In [ ]:
!git clone -q https://github.com/shifat514/medgemma-ner-eval.git
%cd medgemma-ner-eval
!git checkout -q mimic-medication-ner  # branch carrying the MIMIC entrypoint

## 5. Upload the sample file BY HAND

Run the cell, then pick `data/samples/mimic_med_sample.jsonl` from your machine.

No Drive mount, no download — the file goes straight into this VM's ephemeral
disk and disappears when the runtime is recycled.

In [ ]:
import os, shutil
from google.colab import files

os.makedirs('data/samples', exist_ok=True)
DEST = 'data/samples/mimic_med_sample.jsonl'

if os.path.exists(DEST):
    print('already present:', DEST)
else:
    uploaded = files.upload()  # choose mimic_med_sample.jsonl
    name = next(iter(uploaded))
    if name != DEST:
        shutil.move(name, DEST)

# Sanity check WITHOUT printing note text.
import json
recs = [json.loads(l) for l in open(DEST) if l.strip()]
print(f'notes: {len(recs)}')
print(f'gold spans: {sum(len(r["spans"]) for r in recs):,}')
print(f'chars: {sum(len(r["text"]) for r in recs):,}')
assert len(recs) >= 100, 'expected 100 notes — rebuild with: python -m src.build_mimic_sample'

## 6. Smoke test — 5 notes

Finds the memory ceiling before committing to a long run. 5 notes is ~25–30
chunks, so this exercises model load under 4-bit, the chunk loop, JSON parsing,
alignment, merge/dedupe and scoring end to end.

**Watch for:**
- CUDA OOM → lower `--chunk-words` (try 300, then 250).
- `chunks where generation hit max_new_tokens` > 0 in the report → raise
  `MIMIC_MAX_NEW_TOKENS`, or lower `--chunk-words` so each chunk emits fewer
  entities.
- Wall-clock per chunk → multiply by 262 (n=50) or 561 (n=100) to project.

In [ ]:
!python -m src.evaluate_mimic --limit 5

In [ ]:
# Peak VRAM from the smoke run — headroom check before the long runs.
import torch
print('peak allocated: %.2f GB' % (torch.cuda.max_memory_allocated() / 1e9))
print('peak reserved:  %.2f GB' % (torch.cuda.max_memory_reserved() / 1e9))

## 7. Run n=50

~262 chunks. **Results are saved per note as they finish**, to
`outputs/mimic/<tag>/per_note.jsonl`. If Colab disconnects, just re-run this
cell — completed notes are skipped and only the remainder runs. The 5 smoke
notes are already cached and will be reused.

Writes `results/mimic_ner_50.csv` and `results/mimic_ner_50_report.md`.

In [ ]:
!python -m src.evaluate_mimic --n 50

## 8. Run n=100

~561 chunks total, but the 50 notes above are cached, so this only runs the
additional 50 notes (~299 chunks). The n=50 set is the first 50 of the same
seeded draw, so it is a strict subset and the two numbers are comparable.

Writes `results/mimic_ner_100.csv` and `results/mimic_ner_100_report.md`.

In [ ]:
!python -m src.evaluate_mimic --n 100

## 9. Harness ceiling (no GPU, seconds)

Feeds the gold spans back through the identical pipeline. Whatever this loses is
the harness's structural limit — string-matching alignment, chunking, whitespace
tokenization — not model error. Read MedGemma's numbers against these.

In [ ]:
!python -m src.evaluate_mimic --oracle --n 100

## 10. Compare the runs

In [ ]:
import pandas as pd

frames = []
for label in ['50', '100', 'oracle_100']:
    path = f'results/mimic_ner_{label}.csv'
    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        continue
    df.insert(0, 'run', f'n={label}')
    frames.append(df)

combined = pd.concat(frames, ignore_index=True)
print(combined.to_string(index=False))
print('\n--- micro avg across runs ---')
print(combined[combined.entity == 'micro avg'].to_string(index=False))

## 11. Download the results (aggregate only — safe)

These files contain precision/recall/F1/support and configuration only: no note
text, no patient data, no example snippets. Safe to commit.

Do **not** download `outputs/` — it holds per-note run state, and if you passed
`--dump-errors`, per-example dumps that quote note text.

In [ ]:
import glob
from google.colab import files

for path in sorted(glob.glob('results/mimic_ner_*.csv')
                   + glob.glob('results/mimic_ner_*_report.md')):
    print('downloading', path)
    files.download(path)

---

## Tuning notes

**If you hit CUDA OOM:** lower `--chunk-words`. A 400-token chunk of a discharge
summary is ~600 model tokens of input; the T4's 16 GB is comfortable for the
4-bit 4B model, so OOM usually means the *generation* grew long, not the input.

```bash
python -m src.evaluate_mimic --n 50 --chunk-words 250 --overlap-words 50
```

Changing chunk settings changes the run tag, so it starts a **fresh** cache
rather than mixing results computed under different settings.

**If generation keeps hitting the cap:** a 400-token chunk can legitimately hold
30+ entities. Raise the cap:

```bash
MIMIC_MAX_NEW_TOKENS=1536 python -m src.evaluate_mimic --n 50
```

**Resuming:** re-running the same command always resumes. To force a clean rerun,
pass `--no-resume`.

**Before saving this notebook:** `Edit -> Clear all outputs`.